# Diabetic Retinopathy Severity Classification
EfficientNet-B0 fine-tuned on the [APTOS 2019 Blindness Detection](https://www.kaggle.com/c/aptos2019-blindness-detection/data) dataset.

**Before running:** Make sure the runtime is set to GPU — Runtime → Change runtime type → T4 GPU.

## 1. Git Setup

In [1]:
!git config --global user.name "PaarthP89"
!git config --global user.email "paarthp89@gmail.com"

## 2. Clone Repository

In [2]:
import os

REPO_URL = "https://github.com/noteesh/Diabetic-Retinopathy-Severity-Classification.git"
REPO_DIR = "Diabetic-Retinopathy-Severity-Classification"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

Cloning into 'Diabetic-Retinopathy-Severity-Classification'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 42 (delta 10), reused 35 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 2.91 MiB | 30.75 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Diabetic-Retinopathy-Severity-Classification
Working directory: /content/Diabetic-Retinopathy-Severity-Classification


## 3. Install Dependencies

In [3]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.5 MB/s eta 0:00:00


## 4. Just take the L and import the data yourself


its google drive - share the dataset zip to your google drive account, then add
a shortcut to My Drive, then you should be good.

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.makedirs('data', exist_ok=True)
!unzip -q "/content/drive/MyDrive/aptos2019-blindness-detection.zip" -d data/
print("Files:", os.listdir('data'))

Mounted at /content/drive
Files: ['test_images', 'sample_submission.csv', 'train_images', 'train.csv', 'test.csv']


## 6. Resize Photos cause it's taking way too long to train


In [11]:
from PIL import Image
from pathlib import Path
import os

src = Path('data/train_images')
dst = Path('data/train_images_224')
dst.mkdir(exist_ok=True)

for f in src.glob('*.png'):
    img = Image.open(f).convert('RGB')
    img = img.resize((224, 224), Image.LANCZOS)
    img.save(dst / f.name, 'PNG', optimize=False, compress_level=1)

print('Done')

Done


## 7. Train the Model

Adjust `--epochs` and `--batch-size` to taste. T4 GPU handles batch size 32 comfortably at 224px.

In [13]:
!python src/run_improved.py --csv-path data/train.csv --img-dir data/train_images_224 --results-dir results --epochs 20 --batch-size 32 --img-size 224 --lr 1e-4 --weight-decay 1e-4 --label-smoothing 0.1 --dropout 0.3 --num-workers 4

/content/Diabetic-Retinopathy-Severity-Classification/src/run_improved.py:3: SyntaxWarning: invalid escape sequence '\p'
  Format: py -3 src/run_improved.py --csv-path "C:\path\to\train.csv" --img-dir "C:\path\to\train_images"
Using device: cuda
Train: 2563 | Val: 549 | Test: 550
Train class distribution:
diagnosis
0    1263
1     259
2     699
3     135
4     207
Name: count, dtype: int64

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Class weights:
  No DR (0): 0.406
  Mild (1): 1.979
  Moderate (2): 0.733
  Severe (3): 3.797
  Proliferative (4): 2.476
Traina

## 8. Results

In [ ]:
from IPython.display import Image as IPyImage, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

result_plots = [
    ('results/improved_training_curves.png', 'Training Curves'),
    ('results/improved_confusion_matrix.png', 'Confusion Matrix'),
]

for path, title in result_plots:
    if os.path.exists(path):
        fig, ax = plt.subplots(figsize=(12, 5) if 'curves' in path else (8, 7))
        ax.imshow(mpimg.imread(path))
        ax.axis('off')
        ax.set_title(title, fontsize=14, pad=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not found: {path}")

## 9. Save Results to Google Drive (optional)

In [ ]:
# Uncomment to mount Drive and copy results

# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copytree('results', '/content/drive/MyDrive/DR_results', dirs_exist_ok=True)
# print("Results saved to Google Drive.")